# EXP01 — Reprodução do artigo original

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


## 2. Instalar as dependências

Instalamos as bibliotecas usadas nas análises. Se estiver usando um ambiente local já configurado, a instalação pode ser ignorada.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## Enviar e conferir os dados

O EXP01 e o EXP02 usam os mesmos arquivos Parquet processados da pesquisa. Baixe os arquivos a partir de um acesso autorizado à pasta oficial (veja `docs/DADOS.md`). A célula abaixo aceita os arquivos no Colab ou verifica os que já estiverem em `data/processed/` na execução local.

A verificação SHA-256 evita misturar versões diferentes. Ela não comprova que o pré-processamento original esteja correto.


In [ ]:
from pathlib import Path
import hashlib

ARQUIVOS_E_HASHES = {
    "features_treino.parquet": "0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32",
    "features_teste_20201029.parquet": "6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51",
}

pasta_dados = RAIZ / "data" / "processed"
pasta_dados.mkdir(parents=True, exist_ok=True)
faltantes = [nome for nome in ARQUIVOS_E_HASHES if not (pasta_dados / nome).is_file()]

if faltantes:
    try:
        import google.colab  # disponível apenas no Google Colab
    except ImportError as exc:
        raise FileNotFoundError(
            "Arquivos ausentes em data/processed/: " + ", ".join(faltantes)
        ) from exc

    from google.colab import files
    print("Envie os arquivos necessários:", ", ".join(faltantes))
    recebidos = files.upload()
    for nome, conteudo in recebidos.items():
        if nome not in ARQUIVOS_E_HASHES:
            raise ValueError("Arquivo não esperado: " + nome)
        (pasta_dados / nome).write_bytes(conteudo)

for nome, esperado in ARQUIVOS_E_HASHES.items():
    arquivo = pasta_dados / nome
    if not arquivo.is_file():
        raise FileNotFoundError("Arquivo ausente: " + str(arquivo))
    sha = hashlib.sha256()
    with arquivo.open("rb") as entrada:
        for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
            sha.update(bloco)
    encontrado = sha.hexdigest()
    if encontrado != esperado:
        raise ValueError(
            f"SHA-256 diferente em {nome}. Esperado: {esperado}; encontrado: {encontrado}"
        )
    print(f"Arquivo conferido: {nome} ({arquivo.stat().st_size} bytes)")


## 3. Carregar os dados

Precisamos de `features_treino.parquet` e `features_teste_20201029.parquet` em `data/processed/`. Esses arquivos **não** são baixados do Drive privado. Confira `docs/DADOS.md` e não troque os dados por uma amostra sintética para declarar reprodução.

In [ ]:
from pathlib import Path
from ethmix.features import FEATURES
from ethmix.execucao import abrir_dados, verificar_referencia

PASTA = RAIZ / "data" / "processed"
ARQ_TREINO = PASTA / "features_treino.parquet"
ARQ_2020 = PASTA / "features_teste_20201029.parquet"

for arquivo in [ARQ_TREINO, ARQ_2020]:
    print(verificar_referencia(arquivo))

treino = abrir_dados(ARQ_TREINO, ["label", "n_tx", *FEATURES])
teste = abrir_dados(ARQ_2020, ["recipient", "n_tx", "tornado", *FEATURES])
print("Observações de treino:", len(treino), "| Destinatários no teste:", len(teste))

## 4. Reproduzir o experimento

Random Forest com 100 árvores, profundidade máxima 5 e semente 42. O balanceamento é feito no treino. A seleção do teste usa pelo menos dez transações e o alerta corresponde a `score > 0,50`. A validação em cinco divisões é **interna**; o contrato positivo se repete entre treino e validação.

In [ ]:
from ethmix.execucao import treinar, prever

modelo, validacao = treinar(treino, seed=42, min_tx=10)
predicoes = prever(modelo, teste, min_tx=10, limiar=0.50)
alertas = predicoes.loc[predicoes["alerta"]].copy()

print("Média das métricas internas:")
print(validacao[["precision", "recall", "f1"]].mean().round(4))
print("Destinatários elegíveis:", len(predicoes))
print("Alertas:", len(alertas))
print("Contratos conhecidos sinalizados:", alertas["tornado"].notna().sum())
display(alertas[["recipient", "score", "n_tx"]].sort_values("score", ascending=False).head(20))

## 5. Salvar e conferir

A referência histórica é **9 alertas e 3/3 pools conhecidos** em 29/10/2020. Se a execução gerar outro valor, registre a diferença: não mude os parâmetros apenas para alcançar o número publicado.

In [ ]:
from datetime import datetime, timezone
from ethmix.execucao import guardar_arquivos

quando = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
saida = RAIZ / "results" / "generated" / "EXP01" / quando
manifesto = {"experimento": "EXP01", "min_tx": 10, "limiar": ">0.50",
             "seed": 42, "observacoes_treino": len(treino),
             "destinatarios_elegiveis": len(predicoes), "alertas": len(alertas)}
guardar_arquivos(saida, modelo, validacao, alertas, manifesto)
print("Arquivos salvos em:", saida)

## 6. O que podemos concluir?

O número de alertas e a recuperação dos pools conhecidos são resultados observados no recorte. Não permitem calcular precisão populacional nem provar que o detector identifica outros protocolos inéditos.